## Neighbourhood Boundary

In [ ]:
# Load Neighbourhood Boundary
import geopandas as gpd

# Load the shapefile for Kelurahan (neighbourhood) in Jakarta
kelurahan_shp_path = "[R2402] British Academy - Empowering Resilience In a Sinking City/2. Data/1. Big Data/Area of Interest (Source_ Jakarta Satu)/Kelurahan/adm_dki-jakarta_kelurahan.shp"
kelurahan_gdf = gpd.read_file(kelurahan_shp_path)
kelurahan_gdf = kelurahan_gdf[['OBJECTID', 'WADMKD', 'WADMKC', 'WADMKK', 'geometry']]
kelurahan_gdf

In [ ]:
# Load RW (Rukun Warga) shapefile for Jakarta
rw_shp_path = "[R2402] British Academy - Empowering Resilience In a Sinking City/2. Data/1. Big Data/Area of Interest (Source_ Jakarta Satu)/RW/adm_dki-jakarta_rw.shp"
rw_gdf = gpd.read_file(rw_shp_path)
rw_gdf = rw_gdf[['OBJECTID', 'WADMRW', 'WADMKD', 'WADMKC', 'WADMKK', 'geometry']]
rw_gdf

# Socioeconomic Data

## Age by Neighbourhood (Kelurahan)

### Load Podes Data of Age

In [ ]:
import pandas as pd

# Load the socioeconomic data
podes_age_path = "Socioeconomic Statistic/Jumlah Penduduk WNI Provinsi DKI Jakarta Tahun 2016 s.d Tahun 2024 berdasarkan Usia -1.xlsx"

In [ ]:
# List all sheet names in the Excel file
podes_age_list = pd.ExcelFile(podes_age_path)
print(podes_age_list.sheet_names)

In [ ]:
# Load the socioeconomic data based on the sheet
sheet_name = '2016'

podes_age_df = pd.read_excel(podes_age_path, sheet_name=sheet_name)
podes_age_df

### Data Engineering

In [ ]:
# Create a copy to avoid modifying the original DataFrame
podes_age_df = podes_age_df.copy()

# Create a dictionary to map the kelurahan names to their alternative names
alternative_name = {
    'BALEKAMBANG': 'BALE KAMBANG',
    'JATIPULO': 'JATI PULO',
    'KRENDANG': 'KERENDANG',
    'KRAMATJATI': 'KRAMAT JATI',
    'PALMERIAM': 'PAL MERIAM',
    'PINANGRANTI': 'PINANG RANTI',
    'RAWAJATI': 'RAWA JATI',
    'TENGAH': 'KAMPUNG TENGAH'
}

# Replace kelurahan names in podes_age_df using the alternative_name dictionary for those in not_matched
podes_age_df['kelurahan'] = podes_age_df['kelurahan'].replace(alternative_name)

### Compare Data

In [ ]:
# Compare the unique kelurahan codes in the PODES dataset with those in the GeoDataFrame
unique_kelurahan_podes = set(podes_age_df['kelurahan'].dropna().unique())
unique_kelurahan_gdf = set(kelurahan_gdf['WADMKD'].dropna().unique())

# Show the unique rows
print(f"Unique kelurahan in GeoDataFrame: {len(unique_kelurahan_gdf)}")
print(f"Unique kelurahan in PODES: {len(unique_kelurahan_podes)}")

matched = unique_kelurahan_podes & unique_kelurahan_gdf
not_matched = unique_kelurahan_podes - unique_kelurahan_gdf

# Show
print(f"\nUnique matched: {len(matched)}")
print(f"Unique not matched: {len(not_matched)}")

In [ ]:
# # List the matched kelurahan
# print("Matched kelurahan:")
# sorted(matched)

In [ ]:
# List the unmatched kelurahan
print("Unmatched kelurahan:")
sorted(not_matched)

### Join Data

In [ ]:
# Merge the age data with the geodataframe using 'kelurahan' from podes_age_df and 'WADMKD' from kelurahan_gdf
kelurahan_age_2024_gdf = kelurahan_gdf.merge(
    podes_age_df,
    left_on='WADMKD',
    right_on='kelurahan',
    how='left'
)
kelurahan_age_2024_gdf

### Export to Shapefile

In [ ]:
import os

# Save the merged GeoDataFrame to a shapefile
output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)"

# Save the merged GeoDataFrame to a shapefile
output_folder_with_year = os.path.join(output_folder, sheet_name)
os.makedirs(output_folder_with_year, exist_ok=True)

kelurahan_age_2024_gdf.to_file(f"{output_folder_with_year}/socioeconomic_age_{sheet_name}.shp")

## Batch Join for Age

In [ ]:
import os
import pandas as pd

# Load the socioeconomic data
podes_age_path = "Socioeconomic Statistic/Jumlah Penduduk WNI Provinsi DKI Jakarta Tahun 2016 s.d Tahun 2024 berdasarkan Usia -1.xlsx"

# Get all sheet names from the Excel file
sheet_names = pd.ExcelFile(podes_age_path).sheet_names

# Loop through each sheet and process
for sheet in sheet_names:
    # Load data
    df = pd.read_excel(podes_age_path, sheet_name=sheet)
    df = df.copy()
    
    # Create a dictionary to map the kelurahan names to their alternative names
    alternative_name = {
        'BALEKAMBANG': 'BALE KAMBANG',
        'JATIPULO': 'JATI PULO',
        'KRENDANG': 'KERENDANG',
        'KRAMATJATI': 'KRAMAT JATI',
        'PALMERIAM': 'PAL MERIAM',
        'PINANGRANTI': 'PINANG RANTI',
        'RAWAJATI': 'RAWA JATI',
        'TENGAH': 'KAMPUNG TENGAH'
    }

    # Rename kelurahan using the alternative_name dictionary
    df['kelurahan'] = df['kelurahan'].replace(alternative_name)
    
    # Merge with geodataframe
    merged_gdf = kelurahan_gdf.merge(
        df,
        left_on='WADMKD',
        right_on='kelurahan',
        how='left'
    )
    
    # Prepare output folder and save
    output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)"
    output_folder_with_year = os.path.join(output_folder, str(sheet))
    os.makedirs(output_folder_with_year, exist_ok=True)
    merged_gdf.to_file(f"{output_folder_with_year}/socioeconomic_age_{sheet}.shp")

    print(f"socioeconomic_age_{sheet}.shp already exported")

# Print completion message
print("\nJoining process completed.")

## Gender by Neighbourhood (Kelurahan)

### Load Podes Data of Gender

In [ ]:
import pandas as pd

# Load the socioeconomic data
podes_gender_path = "Socioeconomic Statistic\Jumlah Penduduk WNI Provinsi DKI Jakarta Tahun 2015 s.d Tahun 2024 berdasarkan Kelurahan - 1.xlsx"

In [ ]:
# List all sheet names in the Excel file
podes_gender_list = pd.ExcelFile(podes_gender_path)
print(podes_gender_list.sheet_names)

In [ ]:
# Load the socioeconomic data based on the sheet
sheet_name = '2016'

podes_gender_df = pd.read_excel(podes_gender_path, sheet_name=sheet_name)
podes_gender_df

### Data Engineering

In [ ]:
# Rename columns for consistency
podes_gender_df = podes_gender_df.rename(columns={
    'KOTA/KAB. ADM': 'kota',
    'NAMA_KEC': 'kecamatan',
    'NAMA_KEL': 'kelurahan'
})

In [ ]:
# Create a copy to avoid modifying the original DataFrame
podes_gender_df = podes_gender_df.copy()

# Create a dictionary to map the kelurahan names to their alternative names
alternative_name = {
    'BALEKAMBANG': 'BALE KAMBANG',
    'JATIPULO': 'JATI PULO',
    'KALI BARU': 'KALIBARU',
    'KRENDANG': 'KERENDANG',
    'KRAMATJATI': 'KRAMAT JATI',
    'PALMERIAM': 'PAL MERIAM',
    'PINANGRANTI': 'PINANG RANTI',
    'RAWAJATI': 'RAWA JATI',
    'TENGAH': 'KAMPUNG TENGAH'
}

# Replace kelurahan names in podes_gender_df using the alternative_name dictionary for those in not_matched
podes_gender_df['kelurahan'] = podes_gender_df['kelurahan'].replace(alternative_name)

### Compare Data

In [ ]:
# Compare the unique kelurahan codes in the PODES dataset with those in the GeoDataFrame
unique_kelurahan_podes = set(podes_gender_df['kelurahan'].dropna().unique())
unique_kelurahan_gdf = set(kelurahan_gdf['WADMKD'].dropna().unique())

# Show the unique rows
print(f"Unique kelurahan in GeoDataFrame: {len(unique_kelurahan_gdf)}")
print(f"Unique kelurahan in PODES: {len(unique_kelurahan_podes)}")

matched = unique_kelurahan_podes & unique_kelurahan_gdf
not_matched = unique_kelurahan_podes - unique_kelurahan_gdf

# Show
print(f"\nUnique matched: {len(matched)}")
print(f"Unique not matched: {len(not_matched)}")

In [ ]:
# # List the matched kelurahan
# print("Matched kelurahan:")
# sorted(matched)

In [ ]:
# List the unmatched kelurahan
print("Unmatched kelurahan:")
sorted(not_matched)

### Join Data

In [ ]:
# Merge the gender data with the geodataframe using 'kelurahan' from podes_gender_df and 'WADMKD' from kelurahan_gdf
kelurahan_gender_gdf = kelurahan_gdf.merge(
    podes_gender_df,
    left_on='WADMKD',
    right_on='kelurahan',
    how='left'
)
kelurahan_gender_gdf

### Export to Shapefile

In [ ]:
import os

# Save the merged GeoDataFrame to a shapefile
output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)"

# Save the merged GeoDataFrame to a shapefile
output_folder_with_year = os.path.join(output_folder, sheet_name)
os.makedirs(output_folder_with_year, exist_ok=True)

kelurahan_gender_gdf.to_file(f"{output_folder_with_year}/socioeconomic_gender_{sheet_name}.shp")

## Batch Join for Gender

In [ ]:
import os
import pandas as pd

# Load the socioeconomic data
podes_gender_path = "Socioeconomic Statistic\Jumlah Penduduk WNI Provinsi DKI Jakarta Tahun 2015 s.d Tahun 2024 berdasarkan Kelurahan - 1.xlsx"

# Get all sheet names from the Excel file
sheet_names = pd.ExcelFile(podes_gender_path).sheet_names

# Loop through each sheet and process
for sheet in sheet_names:
    # Load data
    df = pd.read_excel(podes_gender_path, sheet_name=sheet)
    df = df.copy()
    
    # Rename columns for consistency
    df = df.rename(columns={
        'KOTA/KAB. ADM': 'kota',
        'NAMA_KEC': 'kecamatan',
        'NAMA_KEL': 'kelurahan'
    })

    # Create a dictionary to map the kelurahan names to their alternative names
    alternative_name = {
        'BALEKAMBANG': 'BALE KAMBANG',
        'JATIPULO': 'JATI PULO',
        'KALI BARU': 'KALIBARU',
        'KRENDANG': 'KERENDANG',
        'KRAMATJATI': 'KRAMAT JATI',
        'PALMERIAM': 'PAL MERIAM',
        'PINANGRANTI': 'PINANG RANTI',
        'RAWAJATI': 'RAWA JATI',
        'TENGAH': 'KAMPUNG TENGAH'
    }

    # Rename kelurahan using the alternative_name dictionary
    df['kelurahan'] = df['kelurahan'].replace(alternative_name)
    
    # Merge with geodataframe
    merged_gdf = kelurahan_gdf.merge(
        df,
        left_on='WADMKD',
        right_on='kelurahan',
        how='left'
    )
    
    # Prepare output folder and save
    output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)"
    output_folder_with_year = os.path.join(output_folder, str(sheet))
    os.makedirs(output_folder_with_year, exist_ok=True)
    merged_gdf.to_file(f"{output_folder_with_year}/socioeconomic_gender_{sheet}.shp")

    print(f"socioeconomic_gender_{sheet}.shp already exported")

# Print completion message
print("\nJoining process completed.")

## Batch Join for Education

In [ ]:
import os
import pandas as pd

# Load the socioeconomic data
podes_education_path = "Socioeconomic Statistic\Jumlah Penduduk WNI Provinsi DKI Jakarta Tahun 2016 s.d Tahun 2024 berdasarkan Pendidikan - 1.xlsx"

# Get all sheet names from the Excel file
sheet_names = pd.ExcelFile(podes_education_path).sheet_names

# Loop through each sheet and process
for sheet in sheet_names:
    # Load data
    df = pd.read_excel(podes_education_path, sheet_name=sheet)
    df = df.copy()
    
    # Create a dictionary to map the kelurahan names to their alternative names
    alternative_name = {
        'BALEKAMBANG': 'BALE KAMBANG',
        'JATIPULO': 'JATI PULO',
        'KRENDANG': 'KERENDANG',
        'KRAMATJATI': 'KRAMAT JATI',
        'PALMERIAM': 'PAL MERIAM',
        'PINANGRANTI': 'PINANG RANTI',
        'RAWAJATI': 'RAWA JATI',
        'TENGAH': 'KAMPUNG TENGAH'
    }

    # Rename kelurahan using the alternative_name dictionary
    df['kelurahan'] = df['kelurahan'].replace(alternative_name)
    
    # Merge with geodataframe
    merged_gdf = kelurahan_gdf.merge(
        df,
        left_on='WADMKD',
        right_on='kelurahan',
        how='left'
    )
    
    # Prepare output folder and save
    output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)"
    output_folder_with_year = os.path.join(output_folder, str(sheet))
    os.makedirs(output_folder_with_year, exist_ok=True)
    merged_gdf.to_file(f"{output_folder_with_year}/socioeconomic_education_{sheet}.shp")

    print(f"socioeconomic_education_{sheet}.shp already exported")

# Print completion message
print("\nJoining process completed.")

## Poverty by Neighbourhood (Kelurahan)

### Load Podes Data of Poverty

In [ ]:
import pandas as pd

# Load the socioeconomic data
podes_poverty_path = "Socioeconomic Statistic/Jumlah Surat Miskin SKTM yang Dikeluarkan Desa-Kelurahan Provinsi DKI Jakarta Tahun 2014, 2018, 2020, 2021, dan 2024.xlsx"

In [ ]:
# List all sheet names in the Excel file
podes_poverty_list = pd.ExcelFile(podes_poverty_path)
print(podes_poverty_list.sheet_names)

In [ ]:
# Load the socioeconomic data based on the sheet
sheet_name = '2014'

'''
For the year of 2014, 2018, 2020, and 2021, the poverty data has an index of R711. For the year of 2024, it has an index of R710.
'''

podes_poverty_df = pd.read_excel(podes_poverty_path, sheet_name=sheet_name)
podes_poverty_df = podes_poverty_df.map(lambda x: x.upper() if isinstance(x, str) else x) # Convert all string values to uppercase (Capitalise)
podes_poverty_df

### Data Engineering

In [ ]:
# Create a copy to avoid modifying the original DataFrame
podes_poverty_df = podes_poverty_df.copy()

# Create a dictionary to map the kelurahan names to their alternative names
alternative_name = {
    'HARAPAN MULYA': 'HARAPAN MULIA',
    'KALI BARU': 'KALIBARU',
    'KALIANYAR': 'KALI ANYAR',
    'KRENDANG': 'KERENDANG',
    'PAL MERIEM': 'PAL MERIAM',
    'PAPANGO': 'PAPANGGO',
    'RAWA SARI': 'RAWASARI',
    'RAWABADAK SELATAN': 'RAWA BADAK SELATAN',
    'RAWABADAK UTARA': 'RAWA BADAK UTARA',
    'RAWAJATI': 'RAWA JATI',
    'SUKA PURA': 'SUKAPURA',
    'TANJUNG PRIUK': 'TANJUNG PRIOK',
    'WIJAYA KESUMA': 'WIJAYA KUSUMA',
    'TENGAH': 'KAMPUNG TENGAH'
}

# Replace kelurahan names in podes_age_df using the alternative_name dictionary for those in not_matched
podes_poverty_df['R104N'] = podes_poverty_df['R104N'].replace(alternative_name)

### Compare Data

In [ ]:
# Compare the unique kelurahan codes in the PODES dataset with those in the GeoDataFrame
unique_kelurahan_podes = set(podes_poverty_df['R104N'].dropna().unique())
unique_kelurahan_gdf = set(kelurahan_gdf['WADMKD'].dropna().unique())

# Show the unique rows
print(f"Unique kelurahan in GeoDataFrame: {len(unique_kelurahan_gdf)}")
print(f"Unique kelurahan in PODES: {len(unique_kelurahan_podes)}")

matched = unique_kelurahan_podes & unique_kelurahan_gdf
not_matched = unique_kelurahan_podes - unique_kelurahan_gdf

# Show
print(f"\nUnique matched: {len(matched)}")
print(f"Unique not matched: {len(not_matched)}")

In [ ]:
# # List the matched kelurahan
# print("Matched kelurahan:")
# sorted(matched)

In [ ]:
# List the unmatched kelurahan
print("Unmatched kelurahan:")
sorted(not_matched)

### Join Data

In [ ]:
# Merge the poverty data with the geodataframe using 'kelurahan' from podes_poverty_df and 'WADMKD' from kelurahan_gdf
kelurahan_poverty_gdf = kelurahan_gdf.merge(
    podes_poverty_df,
    left_on='WADMKD',
    right_on='R104N',
    how='left'
)
kelurahan_poverty_gdf

### Export to Shapefile

In [ ]:
import os

# Save the merged GeoDataFrame to a shapefile
output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)\Poverty Data"

# Save the merged GeoDataFrame to a shapefile
output_folder_with_year = os.path.join(output_folder, sheet_name)
os.makedirs(output_folder_with_year, exist_ok=True)

kelurahan_poverty_gdf.to_file(f"{output_folder_with_year}/socioeconomic_poverty_{sheet_name}.shp")

## Batch Join for Poverty

In [ ]:
import os
import pandas as pd

# Load the socioeconomic data
podes_poverty_path = "Socioeconomic Statistic/Jumlah Surat Miskin SKTM yang Dikeluarkan Desa-Kelurahan Provinsi DKI Jakarta Tahun 2014, 2018, 2020, 2021, dan 2024.xlsx"

# Get all sheet names from the Excel file
sheet_names = pd.ExcelFile(podes_poverty_path).sheet_names

# Loop through each sheet and process
for sheet in sheet_names:
    # Load data
    df = pd.read_excel(podes_poverty_path, sheet_name=sheet)
    df = df.map(lambda x: x.upper() if isinstance(x, str) else x)  # Convert all string values to uppercase (Capitalise)
    df = df.copy()
    
    # Create a dictionary to map the kelurahan names to their alternative names
    alternative_name = {
        'HARAPAN MULYA': 'HARAPAN MULIA',
        'KALI BARU': 'KALIBARU',
        'KALIANYAR': 'KALI ANYAR',
        'KRENDANG': 'KERENDANG',
        'PAL MERIEM': 'PAL MERIAM',
        'PAPANGO': 'PAPANGGO',
        'RAWA SARI': 'RAWASARI',
        'RAWABADAK SELATAN': 'RAWA BADAK SELATAN',
        'RAWABADAK UTARA': 'RAWA BADAK UTARA',
        'RAWAJATI': 'RAWA JATI',
        'SUKA PURA': 'SUKAPURA',
        'TANJUNG PRIUK': 'TANJUNG PRIOK',
        'WIJAYA KESUMA': 'WIJAYA KUSUMA',
        'TENGAH': 'KAMPUNG TENGAH'
    }

    # Rename kelurahan using the alternative_name dictionary
    df['R104N'] = df['R104N'].replace(alternative_name)
    
    # Merge with geodataframe
    merged_gdf = kelurahan_gdf.merge(
        df,
        left_on='WADMKD',
        right_on='R104N',
        how='left'
    )
    
    # Prepare output folder and save
    output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)\Poverty Data"
    output_folder_with_year = os.path.join(output_folder, str(sheet))
    os.makedirs(output_folder_with_year, exist_ok=True)
    merged_gdf.to_file(f"{output_folder_with_year}/socioeconomic_poverty_{sheet}.shp")

    print(f"socioeconomic_poverty_{sheet}.shp already exported")

# Print completion message
print("\nJoining process completed.")

## Cash Assistance by Neighbourhood (Kelurahan)

### Load Podes Data of Cash Assistance

In [ ]:
import pandas as pd

# Load the socioeconomic data
podes_cash_aid_path = "Socioeconomic Statistic/Jumlah Keluarga yang Menerima BLT (Tiga Bulan Pertama) Provinsi DKI Jakarta Tahun 2024.xlsx"

In [ ]:
# List all sheet names in the Excel file
podes_cash_aid_list = pd.ExcelFile(podes_cash_aid_path)
print(podes_cash_aid_list.sheet_names)

In [ ]:
# Load the socioeconomic data based on the sheet
sheet_name = '2024'

'''
All data in the sheet of 2024 has zero values
'''

podes_cash_aid_df = pd.read_excel(podes_cash_aid_path, sheet_name=sheet_name)
podes_cash_aid_df

### Data Engineering

In [ ]:
# Create a copy to avoid modifying the original DataFrame
podes_cash_aid_df = podes_cash_aid_df.copy()

# Create a dictionary to map the kelurahan names to their alternative names
alternative_name = {
    'HARAPAN MULYA': 'HARAPAN MULIA',
    'KALI BARU': 'KALIBARU',
    'KALIANYAR': 'KALI ANYAR',
    'KRENDANG': 'KERENDANG',
    'PAL MERIEM': 'PAL MERIAM',
    'PAPANGO': 'PAPANGGO',
    'RAWA SARI': 'RAWASARI',
    'RAWABADAK SELATAN': 'RAWA BADAK SELATAN',
    'RAWABADAK UTARA': 'RAWA BADAK UTARA',
    'RAWAJATI': 'RAWA JATI',
    'SUKA PURA': 'SUKAPURA',
    'TANJUNG PRIUK': 'TANJUNG PRIOK',
    'WIJAYA KESUMA': 'WIJAYA KUSUMA',
    'TENGAH': 'KAMPUNG TENGAH',
}

# Replace kelurahan names in podes_age_df using the alternative_name dictionary for those in not_matched
podes_cash_aid_df['NAMA_DESA'] = podes_cash_aid_df['NAMA_DESA'].replace(alternative_name)

### Compare Data

In [ ]:
# Compare the unique kelurahan codes in the PODES dataset with those in the GeoDataFrame
unique_kelurahan_podes = set(podes_cash_aid_df['NAMA_DESA'].dropna().unique())
unique_kelurahan_gdf = set(kelurahan_gdf['WADMKD'].dropna().unique())

# Show the unique rows
print(f"Unique kelurahan in GeoDataFrame: {len(unique_kelurahan_gdf)}")
print(f"Unique kelurahan in PODES: {len(unique_kelurahan_podes)}")

matched = unique_kelurahan_podes & unique_kelurahan_gdf
not_matched = unique_kelurahan_podes - unique_kelurahan_gdf

# Show
print(f"\nUnique matched: {len(matched)}")
print(f"Unique not matched: {len(not_matched)}")

In [ ]:
# # List the matched kelurahan
# print("Matched kelurahan:")
# sorted(matched)

In [ ]:
# List the unmatched kelurahan
print("Unmatched kelurahan:")
sorted(not_matched)

### Join Data

In [ ]:
# Merge the cash aid data with the geodataframe using 'kelurahan' from podes_cash_aid_df and 'WADMKD' from kelurahan_gdf
kelurahan_cash_aid_gdf = kelurahan_gdf.merge(
    podes_cash_aid_df,
    left_on='WADMKD',
    right_on='NAMA_DESA',
    how='left'
)
kelurahan_cash_aid_gdf

### Export to Shapefile

In [ ]:
import os

# Save the merged GeoDataFrame to a shapefile
output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)\Cash Assistance Data"

# Save the merged GeoDataFrame to a shapefile
output_folder_with_year = os.path.join(output_folder, sheet_name)
os.makedirs(output_folder_with_year, exist_ok=True)

kelurahan_cash_aid_gdf.to_file(f"{output_folder_with_year}/socioeconomic_cash_assistance_{sheet_name}.shp")

## Informal Settlements by Neighbourhood (Kelurahan)
Residence living in informal settlements

### Load Statistic Data of Slum Area

In [ ]:
import pandas as pd

# Load the socioeconomic data
informal_residence_path = "Socioeconomic Statistic\Data RW Kumuh di Wilayah Provinsi DKI Jakarta.xlsx"

In [ ]:
# List all sheet names in the Excel file
informal_residence_list = pd.ExcelFile(informal_residence_path)
print(informal_residence_list.sheet_names)

In [ ]:
# Load the socioeconomic data based on the sheet
sheet_name = '2023'

informal_residence_df = pd.read_excel(informal_residence_path, sheet_name=sheet_name)
informal_residence_df = informal_residence_df.map(lambda x: x.upper() if isinstance(x, str) else x) # Convert all string values to uppercase (Capitalise)
informal_residence_df

In [ ]:
informal_residence_df.info()

In [ ]:
# Drop unnecessary columns
informal_residence_df = informal_residence_df.drop(columns=['periode_data'])

In [ ]:
# Rename columns by removing the 'klasifikasi_' prefix
informal_residence_df = informal_residence_df.rename(
    columns={col: col.replace('klasifikasi_', '') for col in informal_residence_df.columns if col.startswith('klasifikasi_')}
)
informal_residence_df.head()

### Data Engineering

In [ ]:
# Create a copy to avoid modifying the original DataFrame
informal_residence_df = informal_residence_df.copy()

# Create a dictionary to map the kelurahan names to their alternative names
alternative_name = {
    'HARAPAN MULYA': 'HARAPAN MULIA',
    'JATIPULO': 'JATI PULO',
    'KALIANYAR': 'KALI ANYAR',
    'KEDAUNG KALIANGKE': 'KEDAUNG KALI ANGKE',
    'KRENDANG': 'KERENDANG',
    'PAPANGO': 'PAPANGGO',
    'RAWAJATI': 'RAWA JATI',
}

# Replace kelurahan names using the alternative_name dictionary for those in not_matched
informal_residence_df['kelurahan'] = informal_residence_df['kelurahan'].replace(alternative_name)

### Compare Data

In [ ]:
# Compare the unique kelurahan codes in the PODES dataset with those in the GeoDataFrame
unique_kelurahan_podes = set(informal_residence_df['kelurahan'].dropna().unique())
unique_kelurahan_gdf = set(kelurahan_gdf['WADMKD'].dropna().unique())

# Show the unique rows
print(f"Unique kelurahan in GeoDataFrame: {len(unique_kelurahan_gdf)}")
print(f"Unique kelurahan in PODES: {len(unique_kelurahan_podes)}")

matched = unique_kelurahan_podes & unique_kelurahan_gdf
not_matched = unique_kelurahan_podes - unique_kelurahan_gdf

# Show
print(f"\nUnique matched: {len(matched)}")
print(f"Unique not matched: {len(not_matched)}")

In [ ]:
# # List the matched kelurahan
# print("Matched kelurahan:")
# sorted(matched)

In [ ]:
# List the unmatched kelurahan
print("Unmatched kelurahan:")
sorted(not_matched)

### Join Data

In [ ]:
# Merge the cash aid data with the geodataframe using 'kelurahan' from podes_cash_aid_df and 'WADMKD' from kelurahan_gdf
kelurahan_informal_settlement_gdf = kelurahan_gdf.merge(
    informal_residence_df,
    left_on='WADMKD',
    right_on='kelurahan',
    how='left'
)
kelurahan_informal_settlement_gdf

In [ ]:
# Drop the specified columns
kelurahan_informal_settlement_gdf = kelurahan_informal_settlement_gdf.drop(columns=['wilayah', 'kecamatan', 'kelurahan'])

# Fill NaN values with 0
kelurahan_informal_settlement_gdf = kelurahan_informal_settlement_gdf.fillna(0)

kelurahan_informal_settlement_gdf.head()

In [ ]:
# Convert specified columns to integer type
cols_to_int = ['OBJECTID', 'berat', 'sedang', 'ringan', 'sangat_ringan', 'total']
kelurahan_informal_settlement_gdf[cols_to_int] = kelurahan_informal_settlement_gdf[cols_to_int].astype(int)

In [ ]:
kelurahan_informal_settlement_gdf.info()

In [ ]:
kelurahan_informal_settlement_gdf.sample(5)

### Export to Shapefile

In [ ]:
import os

# Save the merged GeoDataFrame to a shapefile
output_folder = r"[R2402] British Academy - Empowering Resilience In a Sinking City\2. Data\1. Big Data\Socio-Economic Data (Source_ Potensi Desa, BPS)\Residence Living in Informal Settlements"

# Save the merged GeoDataFrame to a shapefile
output_folder_with_year = os.path.join(output_folder, sheet_name)
os.makedirs(output_folder_with_year, exist_ok=True)

kelurahan_informal_settlement_gdf.to_file(f"{output_folder_with_year}/socioeconomic_informal_residence_{sheet_name}.shp")